# Stance Deep Learning

Train and evaluate TextCNN and bidirectional LSTM here. Reports and confusion matrices stay in the notebook.


## Load data and set training parameters

Keep the stratified 80/20 seed-42 split and model settings used by the earlier deep learning scripts.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader

from utils.deep_learning_models import (
    BiLSTMTextClassifier,
    TextCNNClassifier,
    TextDataset,
    build_vocabulary,
    compute_metrics,
    predict,
    train_one_epoch,
)

DATA_PATH = "data/preprocessed/stance_preprocessed.csv"
TEXT_COLUMN = "combined_text"
LABEL_COLUMN = "stance_label"
LABEL_NAMES = ["AGREE","DISAGREE","DISCUSS","UNRELATED"]
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.001

frame = pd.read_csv(DATA_PATH).dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
texts = frame[TEXT_COLUMN].astype(str).tolist()
label_to_id = {name: index for index, name in enumerate(LABEL_NAMES)}
labels = frame["stance_label"].astype(str).str.upper().map(label_to_id)
if labels.isna().any():
    raise ValueError("Unknown stance label in preprocessed data")
labels = labels.astype(int).tolist()

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)
vocabulary = build_vocabulary(train_texts, MAX_VOCAB_SIZE)
train_data = TextDataset(
    train_texts, train_labels, vocabulary, MAX_SEQUENCE_LENGTH
)
test_data = TextDataset(
    test_texts, test_labels, vocabulary, MAX_SEQUENCE_LENGTH
)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training rows:", len(train_data), "Evaluation rows:", len(test_data))


## Train and evaluate

The two model classes represent their neural networks. The training loop, metrics, reports, and plots are visible here.


In [ ]:
models = {
    "text_cnn": TextCNNClassifier(
        vocab_size=len(vocabulary),
        embedding_dim=EMBEDDING_DIM,
        num_classes=len(LABEL_NAMES),
    ),
    "bidirectional_lstm": BiLSTMTextClassifier(
        vocab_size=len(vocabulary),
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=len(LABEL_NAMES),
    ),
}
results = []
trained_models = {}

for model_name, model in models.items():
    model = model.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    epoch_losses = []
    for epoch in range(EPOCHS):
        loss = train_one_epoch(
            model, train_loader, loss_function, optimizer, device
        )
        epoch_losses.append(loss)
        print(model_name, "epoch", epoch + 1, "loss", round(loss, 4))

    true_labels, predictions, probabilities = predict(
        model, test_loader, device
    )
    metrics = compute_metrics(
        true_labels, predictions, probabilities, LABEL_NAMES
    )
    results.append({
        "model": model_name,
        "final_train_loss": epoch_losses[-1],
        **metrics,
    })
    trained_models[model_name] = model

    print(model_name)
    print(classification_report(
        true_labels,
        predictions,
        labels=list(range(len(LABEL_NAMES))),
        target_names=LABEL_NAMES,
        zero_division=0,
    ))
    ConfusionMatrixDisplay.from_predictions(
        true_labels,
        predictions,
        labels=list(range(len(LABEL_NAMES))),
        display_labels=LABEL_NAMES,
        cmap="Blues",
    )
    plt.show()

comparison = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(comparison[["model", "accuracy", "macro_f1", "weighted_f1", "roc_auc"]])


## Interpretation

Compare macro F1 and the DISAGREE class across both models. These models are research baselines; the Streamlit app uses the saved DistilBERT model.
